In [3]:
import polars as pl
import time
import tracemalloc
import matplotlib.pyplot as plt
import numpy as np

csv_path = '/home/montes/proyectos/paralela/polar_vrs_pandas/taxi_filtrado.parquet'

# Comparación Eager vs Lazy

In [4]:
# Eager (read_csv)
tracemalloc.start()
t0 = time.time()
df_eager = pl.read_csv(csv_path)
df_eager = df_eager.filter((pl.col('trip_distance') > 0) & (pl.col('total_amount') > 0))
df_eager = df_eager.with_columns(pl.col('tpep_pickup_datetime').str.to_datetime().dt.hour().alias('hora'))
df_eager = df_eager.group_by('hora').agg(pl.col('total_amount').mean())
t_eager = time.time() - t0
mem_eager = tracemalloc.get_traced_memory()[1] / (1024**2)
tracemalloc.stop()

# Lazy (scan_csv + collect)
tracemalloc.start()
t0 = time.time()
df_lazy = (
    pl.scan_csv(csv_path)
    .filter((pl.col('trip_distance') > 0) & (pl.col('total_amount') > 0))
    .with_columns(pl.col('tpep_pickup_datetime').str.to_datetime().dt.hour().alias('hora'))
    .group_by('hora')
    .agg(pl.col('total_amount').mean())
    .collect()
)
t_lazy = time.time() - t0
mem_lazy = tracemalloc.get_traced_memory()[1] / (1024**2)
tracemalloc.stop()

print(f"Eager: {t_eager:.3f}s | {mem_eager:.1f} MB")
print(f"Lazy:  {t_lazy:.3f}s | {mem_lazy:.1f} MB")
print(f"\nSpeedup: {t_eager/t_lazy:.2f}x")
print(f"Ahorro memoria: {mem_eager - mem_lazy:.1f} MB")

ComputeError: invalid utf-8 sequence